[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abhijeet1520/langgraph-workshop/blob/main/notebooks/level-1-basics/02_simple_graph.ipynb)

# Simple Graph

**Level 1** · Part of the [LangGraph Workshop](https://github.com/Abhijeet1520/langgraph-workshop).

> **Source:** migrated from `snehotosh/langgraph-notebooks` ([original](https://github.com/snehotosh/langgraph-notebooks/blob/main/2_simple_graph.ipynb)) — see [PROVENANCE.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/PROVENANCE.md).

**Provider setup:** set `OPENAI_API_KEY` / `OPENAI_API_BASE` / `OPENAI_MODEL` first — [docs/setup/providers.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/setup/providers.md). The first cells install requirements and verify the connection.

## A Simple Graph - No LLM

In [ ]:
# --- Workshop provider config (edit ONLY these 3 values) ---------------------
import os

os.environ.setdefault("OPENAI_API_KEY", "")   # <- paste your key
os.environ.setdefault("OPENAI_API_BASE", "")  # <- paste your base URL
os.environ.setdefault("OPENAI_MODEL", "openai/gpt-4.1-mini")  # <- your model id

# Verify
for _v in ("OPENAI_API_KEY", "OPENAI_API_BASE", "OPENAI_MODEL"):
    print(_v, "=", "set" if os.getenv(_v) else "**MISSING — see docs/setup/providers.md**")
# -----------------------------------------------------------------------------

In [ ]:
from langchain_openai import ChatOpenAI
import os

def make_llm(**overrides):
    """Provider-agnostic LLM factory used across the workshop."""
    cfg = dict(
        model=os.environ["OPENAI_MODEL"],
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.environ["OPENAI_API_BASE"],
        temperature=0,
    )
    cfg.update(overrides)
    return ChatOpenAI(**cfg)

llm = make_llm()

# Connection check (1 token)
resp = llm.invoke("Say OK")
print("✅ LLM reachable:", resp.content[:60])


In [ ]:
from dotenv import load_dotenv
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, Literal
import random
from IPython.display import display, Image

In [ ]:
class AgentState(TypedDict):
    message: str

In [ ]:
def node_greet(state: AgentState) -> AgentState:
    state["message"] = state["message"] + " I am"
    return state

In [ ]:
def node_happy(state: AgentState) -> AgentState:
    state["message"] = state["message"] + " happy!"
    return state

In [ ]:
def node_sad(state: AgentState) -> AgentState:
    state["message"] = state["message"] + " sad!"
    return state

In [ ]:
random.random()

In [ ]:
def decide_mood(state:AgentState) -> Literal["node_happy", "node_sad"]:
    rand_num = random.random()
    if rand_num < 0.5:
        return "node_happy"
    else:
        return "node_sad"

In [ ]:
graph = StateGraph(AgentState)
#graph.add_node("node_greet", node_greet)
graph.add_node("node_happy", node_happy)
graph.add_node("node_sad", node_sad)

graph.add_conditional_edges(START,
                            decide_mood,
                            {
                                "node_happy": "node_happy",
                                "node_sad": "node_sad"
                            })
graph.add_edge("node_happy", END)
graph.add_edge("node_sad", END)
app = graph.compile()

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
app.invoke({"message": "Hi, this is Snehotosh"})